<a href="https://colab.research.google.com/github/DerrianHarris/Sports-Bot/blob/main/Sports_Bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Global Paramaters { run: "auto" }
URL = 'https://kenpom.com'
folder_path = '/content/drive/MyDrive/SportsBot' #@param {type: 'string'}
strformat = "{:.2f}"

# Setup

In [ ]:
#@title Package Install
%%shell
sudo apt --fix-broken install
python --version
#pip install undetected-chromedriver lxml webdriver_manager selenium_stealth selenium python-dateutil tqdm 'keras-tuner<=1.3.5' beautifulsoup4 ipdb keras
pip install kenpompy
# Install chromium and chromium-driver
#apt-get update
#apt install chromium-chromedriver


#sudo apt -y update
#sudo apt install -y wget curl unzip
#wget http://archive.ubuntu.com/ubuntu/pool/main/libu/libu2f-host/libu2f-udev_1.1.4-1_all.deb
#dpkg -i libu2f-udev_1.1.4-1_all.deb
#wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
#dpkg -i google-chrome-stable_current_amd64.deb
#CHROME_DRIVER_VERSION=`curl -sS chromedriver.storage.googleapis.com/LATEST_RELEASE`
#wget -N https://chromedriver.storage.googleapis.com/$CHROME_DRIVER_VERSION/chromedriver_linux64.zip -P /tmp/
#unzip -o /tmp/chromedriver_linux64.zip -d /tmp/
#chmod +x /tmp/chromedriver
#mv /tmp/chromedriver /usr/local/bin/chromedriver

#chromedriver --version

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
0 upgraded, 0 newly installed, 0 to remove and 29 not upgraded.
Python 3.11.11
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.7/99.7 kB 3.2 MB/s eta 0:00:00


In [ ]:
#@title Google Drive Connection

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#@title Imports

import sys
import json
import re
import traceback
import pdb
import pandas as pd


from os.path import exists as file_exists

from zipfile import ZipFile

from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
import undetected_chromedriver as uc
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from selenium_stealth import stealth

from bs4 import BeautifulSoup

from datetime import datetime
from dateutil import parser
from dateutil.tz import gettz

from tqdm.auto import tqdm

import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow import keras
from keras import layers
import keras_tuner as kt

from time import sleep
import ipdb

ModuleNotFoundError: No module named 'selenium'

In [ ]:
#@title Functions

def init():
  options = webdriver.ChromeOptions()
  #options.add_experimental_option('excludeSwitches', ['enable-logging'])
  options.add_argument("--headless")
  options.add_argument('--no-sandbox')
  options.add_argument('--disable-dev-shm-usage')
  options.add_argument("start-maximized")
  s=Service(ChromeDriverManager().install())
  driver = webdriver.Chrome(service=s,options=options)
  stealth(driver,
        languages=["en-US", "en"],
        vendor="Google Inc.",
        platform="Win32",
        webgl_vendor="Intel Inc.",
        renderer="Intel Iris OpenGL Engine",
        fix_hairline=True,
  )
  return driver

def refresh_driver():
  driver = init()
  login(driver)
  return driver

def mk_int(s):
  s = s.strip()
  return int(s) if s else 0

def login(driver):
  soup = goToURL(driver,URL)
  sleep(5)
  has_login_box = soup.find_all('div',id='login-box')

  if not has_login_box:
    print("Already logged in!")
    return
  !cp /content/drive/MyDrive/SportsBot/config.py /content
  import config
  print("Logging in....")
  email=driver.find_element(By.NAME,"email")
  password=driver.find_element(By.NAME,"password")
  submit=driver.find_element(By.NAME,"submit")
  remember=driver.find_element(By.NAME,"remember")



  email.send_keys(config.EMAIL)
  password.send_keys(config.PASSWORD)
  remember.click()
  submit.click()
  soup = goToURL(driver,URL)
  has_login_box = soup.find_all('div',id='login-box')

  if not has_login_box:
    print("Logged In!")
    return
  raise RuntimeError("Login Failed!")


def goToURL(driver,url):
    driver.get(url)
    return parseHTML(getPageHTML(driver))

def getPageHTML(browser):
  return browser.page_source

def parseHTML(html):
  return BeautifulSoup(html, 'lxml')

def getFullLink(partial):
  return f"{URL}/{partial}"
def getAllTeamsPageLink(driver,year):
  try:
      print('Getting links for all teams....')
      soup = goToURL(driver,getFullLink(f'index.php?y={year}'))
      rows = [getFullLink(f"{row.a['href']}") for row in soup.find_all('td',class_='next_left')]#driver.find_elements(By.XPATH,"//td[contains(@class,'next_left')]/a")
      return rows
  except BaseException  as e:
      print(f"Cannot get team links! {e}")

def getDataFromGame(driver,game_link,location,team_name):
  soup = goToURL(driver,game_link)
  game_info = soup.find('span',class_='game-info')#driver.find_element(By.XPATH,"//div[contains(@id,'half-column3')]").text.split('·')

  pdb.set_trace()
  date_info = game_info.find(href=re.compile("fanmatch.php"))

  date = date_info.text if date_info else ''
  time = game_info.find_all(string=True,recursive=False)[0].replace('·','').strip()
  try:
    date_time = f"{date} {time}"
    date_time = parser.parse(date_time,tzinfos={"ET": gettz("America/New_York")})
  except:
    date_time = ''

  arena_info = game_info.find(href=re.compile("maps.google.com"))
  arena = arena_info.text.strip() if arena_info else ''

  game = {}
  game["location"] = location
  game["arena"] = arena
  game["date_time"] = date_time.isoformat() if isinstance(date_time,datetime) else ''

  team_names = [name.b.text.strip() for name in soup.find_all('div',class_='teamnav')]
  team_ranks = [rank.text.strip() for rank in soup.find('h2', class_='scoreline').find_all('span', class_='team-rank',string=True)][::-1]

  records = [record.text.strip() for record in game_info.find_all(style='font-size:14px;')]
  box_wrappers = soup.find_all('table',class_='box-table')
  game["teams"] = [{},{}]
  for i in range(2):
      game["teams"][i]["name"] = team_names[i]

      rank = team_ranks[i]
      game["teams"][i]["rank"] = rank.strip() if rank.strip().lower() != 'nr' else -1
      game["teams"][i]["ovr_wins"] = 0
      game["teams"][i]["ovr_loses"] = 0
      game["teams"][i]["conf_wins"] = 0
      game["teams"][i]["conf_loses"] = 0

      has_record = team_names[i][1] == True
      record_len = len(records)
      if has_record:

        record_num = i
        if record_len < 2:
          record_num = 0

        record = records[record_num]
        win_loses = [rcd.split('-') for rcd in record[1:len(record) - 1].split(',')]
        win_loses = sum(win_loses, [])

        count = len(win_loses) / 2

        game["teams"][i]["ovr_wins"] = win_loses[0] if count > 0 else 0
        game["teams"][i]["ovr_loses"] = win_loses[1] if count > 0 else 0
        game["teams"][i]["conf_wins"] = win_loses[2] if count > 1 else 0
        game["teams"][i]["conf_loses"] = win_loses[3] if count > 1 else 0

      players_rows = box_wrappers[i].tbody.find_all('tr',class_=lambda x: x != 'bottom' and x != 'bottom2')
      team_stats_raw = [stat.find('b') if stat.find('b') != None else '' for stat in players_rows[len(players_rows)-1].find_all("td")]
      team_stats = [stat.text if type(stat) != str else stat for stat in team_stats_raw]

      game["teams"][i]['min'] = mk_int(team_stats[3])
      game["teams"][i]['pts'] = mk_int(team_stats[6])

      two_pts = team_stats[7].split('-')
      game["teams"][i]['fgm'] = mk_int(two_pts[0])
      game["teams"][i]['fga'] = mk_int(two_pts[1])

      three_pts = team_stats[8].split('-')
      game["teams"][i]['3fgm'] = mk_int(three_pts[0])
      game["teams"][i]['3fga'] = mk_int(three_pts[1])

      ft_pts = team_stats[9].split('-')
      game["teams"][i]['ftm'] = mk_int(ft_pts[0])
      game["teams"][i]['fta'] = mk_int(ft_pts[1])

      game["teams"][i]['oreb'] = mk_int(team_stats[10])
      game["teams"][i]['dreb'] = mk_int(team_stats[11])
      game["teams"][i]['ast'] = mk_int(team_stats[12])
      game["teams"][i]['to'] = mk_int(team_stats[13])
      game["teams"][i]['blk'] = mk_int(team_stats[14])
      game["teams"][i]['stl'] = mk_int(team_stats[15])
      game["teams"][i]['pf'] = mk_int(team_stats[16])

      game["teams"][i]["players"] = []
      for j in range(len(players_rows)-3):
          player_stats_raw = [stat.text.replace('\n','') for stat in players_rows[j].find_all("td")]
          player_stats = [stat for stat in player_stats_raw if stat != 'Team']

          name = player_stats[2]

          if name == '':
            continue



          player_info = player_stats[0].split(' ')

          player = {}
          player['height'] = ''
          player['weight'] = 0
          player['year'] = ''

          if len(player_info) > 2:
            player['height'] = player_info[0]
            player['weight'] = mk_int(player_info[1])
            player['year'] = player_info[2]

          player['number'] = mk_int(player_stats[1])
          player['name'] = name.replace('MVP','')
          player['min'] = mk_int(player_stats[3])
          player['ORtg'] = mk_int(player_stats[4])
          player['usage_per'] = mk_int(player_stats[5])
          player['pts'] = mk_int(player_stats[6])

          two_pts = player_stats[7].split('-')
          player['fgm'] = mk_int(two_pts[0])
          player['fga'] = mk_int(two_pts[1])

          three_pts = player_stats[8].split('-')
          player['3fgm'] = mk_int(three_pts[0])
          player['3fga'] = mk_int(three_pts[1])

          ft_pts = player_stats[9].split('-')
          player['ftm'] = mk_int(ft_pts[0])
          player['fta'] = mk_int(ft_pts[1])

          player['oreb'] = mk_int(player_stats[10])
          player['dreb'] = mk_int(player_stats[11])
          player['ast'] = mk_int(player_stats[12])
          player['to'] = mk_int(player_stats[13])
          player['blk'] = mk_int(player_stats[14])
          player['stl'] = mk_int(player_stats[15])
          player['pf'] = mk_int(player_stats[16])

          game["teams"][i]["players"].append(player)
  formatData(team_name,game)
  return game

def getTeamData(driver,link,team_data):
  soup = goToURL(driver,link)
  if not team_data:
    team_data = {'games':[]}

  title = soup.find('div',id='title-container')#driver.find_element(By.XPATH,"//div[contains(@id,'title-container')]")
  schedule = soup.find('div',id='schedule')
  header = title.h5

  #OWN_TEXT_SCRIPT = "if(arguments[0].hasChildNodes()){var r='';var C=arguments[0].childNodes;for(var n=0;n<C.length;n++){if(C[n].nodeType==Node.TEXT_NODE){r+=' '+C[n].nodeValue}}return r.trim()}else{return arguments[0].innerText}"
  team_data["name"]  = header.find_all(string=True, recursive=False)[0].strip()#driver.execute_script(OWN_TEXT_SCRIPT, header)

  record_info = header.find_all('span',class_='rank')
  rank = record_info[0].text
  record = record_info[1].text

  team_data["rank"] = mk_int(rank)
  win_loses = record[1:len(record) - 1].split('-')
  team_data["wins"] = mk_int(win_loses[0])
  team_data["losses"] = mk_int(win_loses[1])
  team_data["games_played"] =  team_data["wins"] + team_data["losses"]

  team_data["mascot"] = title.a.contents[0].strip()
  team_data["location"] = title.contents[4].replace('·','').strip()

  #coach_span = driver.find_element(By.XPATH,"//span[contains(@class,'coach')]//a")
  #team_data["head_coach"] = coach_span.text

  coach_span = title.find('span',class_='coach').a
  team_data["head_coach"] = coach_span.text.strip()

  other_info_span = title.find('span',class_='otherinfo').a#driver.find_element(By.XPATH,"//span[contains(@class,'otherinfo')]//a")

  team_data["conference"] = other_info_span.text.strip()

  game_links = schedule.find_all(href=re.compile("box.php"))#driver.find_elements(By.XPATH,"//tr[@class = 'w' or @class = 'l']//a[contains(@href,'box.php')]")
  locations = schedule.find_all(href=re.compile("maps.google"))#driver.find_elements(By.XPATH,"//tr[@class = 'w' or @class = 'l']//a[contains(@href,'maps.google')]")

  game_links = [getFullLink(link['href']) for link in game_links]
  locations = [location.text for location in locations]

  zip_link_loc = list(zip(game_links,locations))
  total_games = len(zip_link_loc)
  start_index = len(team_data["games"])
  for (link,location) in zip_link_loc[start_index:]:
    team_data["games"].append(getDataFromGame(driver,link,location,team_data["name"]))

  return (team_data,start_index < total_games)

def getAllTeamData(driver,team_links,file_names,YEAR):

  link_count = len(team_links)
  data = loadData(link_count,file_names)
  print('Scrapping team links for game data....')
  count = 0
  save_interval = 15
  updated = False
  for i in tqdm(iterable=range(link_count),leave=True,total=link_count,position=0,desc=f"{YEAR} Teams",ascii=True):
      link = team_links[i]
      data[i], data_updated = getTeamData(driver,link,data[i])
      count+=1
      if data_updated and count > save_interval:
        saveData(data,file_names)
        count = 0
        updated = True
  return (data,updated)

def loadData(link_count,file_names):
  try:
    file_name,zip_file_path = file_names
    with ZipFile(f'{zip_file_path}.zip', 'r') as myzip:
        return json.loads(myzip.read(f'{file_name}.json'))
  except:
    return [None] * link_count

def saveData(data,file_names):
  file_name,zip_file_path = file_names
  with ZipFile(f'{zip_file_path}.zip', 'w') as myzip:
    myzip.writestr(f'{file_name}.json',json.dumps(data))

def formatData(team_name,game):
  if game['teams'][0]['name'] != team_name:
    game['teams'][0], game['teams'][1] = game['teams'][1],game['teams'][0]

def reformatTeamData(team_data,YEAR):
  teams = {}
  for team in team_data:
    games = {}
    if not team:
      continue
    for game in team['games']:
      game_teams = {}
      for game_team in game['teams']:
        game_teams[game_team['name']] = game_team
      game['teams'] = game_teams
      games[game['date_time']] = game
    team['games'] = games
    teams[f"{team['name']}_{str(YEAR)}"] = team
  return teams

def reformGameData(team_data,YEAR):
  games = []
  for team_name in team_data:
    for game_date in team_data[team_name]['games']:

      game = team_data[team_name]['games'][game_date]
      teams = list(game['teams'].values())

      if len(teams) < 2:
        continue

      data = {}
      data['date'] = game['date_time']
      data['location'] = game['location']

      for i in range(2):
        j = i+1
        data[f"t{j}_name"] = f"{teams[i]['name']}_{YEAR}"
        data[f"t{j}_wins"] = teams[i]['ovr_wins']
        data[f"t{j}_loses"] = teams[i]['ovr_loses']
        data[f"t{j}_rank"] = teams[i]['rank']
        data[f"t{j}_pts"] = teams[i]['pts']
        data[f"t{j}_fgm"] = teams[i]['fgm']
        data[f"t{j}_fga"] = teams[i]['fga']
        data[f"t{j}_3fgm"] = teams[i]['3fgm']
        data[f"t{j}_3fga"] = teams[i]['3fga']
        data[f"t{j}_ftm"] = teams[i]['ftm']
        data[f"t{j}_fta"] = teams[i]['fta']
        data[f"t{j}_oreb"] = teams[i]['oreb']
        data[f"t{j}_dreb"] = teams[i]['dreb']
        data[f"t{j}_ast"] = teams[i]['ast']
        data[f"t{j}_to"] = teams[i]['to']
        data[f"t{j}_blk"] = teams[i]['blk']
        data[f"t{j}_stl"] = teams[i]['stl']
        data[f"t{j}_pf"] = teams[i]['pf']
      games.append(data)
  game_data = pd.DataFrame(games)
  game_data['date'] = pd.to_datetime(game_data['date'],utc=True,errors='raise')
  game_data = game_data.drop(game_data[game_data['date']=='NaT'].index)
  game_data = game_data.sort_values('date')
  game_data = game_data.join(pd.get_dummies(game_data['location']))
  game_data = game_data.drop(['t1_wins','t1_loses','t2_wins','t2_loses','location'],axis=1)

  game_data['t1_rank'] = game_data['t1_rank'].replace('NR',-1).astype('int64')
  game_data['t2_rank'] = game_data['t2_rank'].replace('NR',-1).astype('int64')

  max_rank = game_data[['t1_rank','t2_rank']].max(axis=0)
  max_rank = max_rank[0] if max_rank[0] > max_rank[1] else max_rank[1]

  game_data['t1_rank']= game_data['t1_rank'].replace(-1, max_rank+1)
  game_data['t2_rank']= game_data['t2_rank'].replace(-1, max_rank+1)

  game_data = game_data.dropna()
  game_data = game_data.drop_duplicates()
  return game_data

def getOutputData(team_data, game_data):
  output = pd.DataFrame()
  dataframe = pd.DataFrame()
  for team_name in tqdm(iterable=team_data,leave=True,desc=f'Generating {YEAR} AVG Games'):
    data = game_data[game_data.t1_name == team_name]
    data = data.sort_values('date')
    if data.shape[0] > prev_games_to_avg:
      for i in range(prev_games_to_avg,data.shape[0]):
        prev_games = data.iloc[i-prev_games_to_avg:i]
        prev_games = prev_games.drop(['date','t1_name'], axis=1)
        prev_game_avg = prev_games.agg('mean',numeric_only=True)
        prev_game_avg = prev_game_avg.rename(lambda x: x+'_avg')
        prev_game_avg['pts'] = data.iloc[i]['t1_pts']
        prev_game_avg['t1_name'] = data.iloc[i]['t1_name']
        prev_game_avg['t2_name'] = data.iloc[i]['t2_name']
        prev_game_avg['t1_rank'] = data.iloc[i]['t1_rank']
        prev_game_avg['t2_rank'] = data.iloc[i]['t2_rank']
        prev_game_avg['Home'] = data.iloc[i]['Home']
        prev_game_avg['Away'] = data.iloc[i]['Away']
        prev_game_avg['Neutral'] = data.iloc[i]['Neutral']
        prev_game_avg['Semi-Away'] = data.iloc[i]['Semi-Away']
        prev_game_avg['Semi-Home'] = data.iloc[i]['Semi-Home']
        prev_game_avg['date'] = data.iloc[i]['date']
        dataframe = dataframe.append(prev_game_avg, ignore_index=True)
  data = dataframe.iterrows()
  for index,row in tqdm(iterable=data,total=dataframe.shape[0],leave=True,desc=f'Merging {YEAR} AVG Games'):
    date = row['date']
    t2_name = row['t2_name']

    t1_game = row.to_frame().T.convert_dtypes()
    t2_game = dataframe[(dataframe.date == date) & (dataframe.t1_name == t2_name)].convert_dtypes()

    merged = t1_game.merge(t2_game, suffixes=('_left', '_right'),on='date')
    output = output.append(merged,ignore_index=True)
  output = output.reindex(sorted(output.columns), axis=1)
  return output

def getFileNames(YEAR):
  file_name = f"ncaa_basketball_stats_{YEAR}"
  zip_file_path = f"{folder_path}/{file_name}"
  return (file_name,zip_file_path)

def setPandasDisplayLimit(limit):
  pd.set_option("display.max_rows", limit, "display.max_columns", None)


# Data gathering/formating

In [ ]:
from numpy.lib.shape_base import column_stack
#@title Data Gathering Parameters
YEARS =  [2025,2024,2023,2022,2021,2020]#@param {type: 'raw'}
prev_games_to_avg = 10 #@param {type: 'integer'}
SCRAPE_DATA = True #@param {type: 'boolean'}

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
# setPandasDisplayLimit(10)
!cp /content/drive/MyDrive/SportsBot/config.py /content
import config
from kenpompy.utils import login
import kenpompy.summary as kp_summary
import kenpompy.team as kp_teams
browser = login(config.EMAIL, config.PASSWORD)

for season in YEARS:
  teams = kp_teams.get_valid_teams(browser, season)
  for team in teams:
    schedule = kp_teams.get_schedule(browser, team, season)
    display(schedule)
    break
  break

# team_data = {}
# updated = {}
# for YEAR in YEARS:
#   updated[YEAR] = False

# if SCRAPE_DATA:
#   driver = refresh_driver()
#   for YEAR in YEARS:
#     done = False
#     while not done:
#       try:
#         team_links = getAllTeamsPageLink(driver,YEAR)
#         team_data[YEAR],updated[YEAR] = getAllTeamData(driver,team_links,getFileNames(YEAR),YEAR)
#       except Exception as err:
#         print(traceback.format_exc())
#         driver = refresh_driver()
#         sys.exit()
#       else:
#         done = True



#   print("\nDone!")
# else:

#   for YEAR in tqdm(iterable=YEARS,desc=f'Loading Team data: '):
#     team_data[YEAR] = loadData(0,getFileNames(YEAR))

# year_game_data = {}

# output = pd.DataFrame()

# for YEAR in team_data:
#   team_data[YEAR] = reformatTeamData(team_data[YEAR],YEAR)
#   year_game_data[YEAR] = reformGameData(team_data[YEAR],YEAR)

#   year_output = None

#   output_file_name = f"ncaa_basketball_stats_output_{str(YEAR)}_{str(prev_games_to_avg)}"



#   output_file_name_zip = f"{output_file_name}.zip"
#   output_file_name_csv = f"{output_file_name}.csv"

#   output_file_path = f"{folder_path}/{output_file_name_zip}"
#   if file_exists(output_file_path) and not updated[YEAR]:
#     year_output = pd.read_csv(output_file_path)
#     print(f'Loaded {YEAR} AVG from: {output_file_path}')
#   else:
#     year_output = getOutputData(team_data[YEAR],year_game_data[YEAR])
#     compression_opts = dict(method='zip',archive_name=output_file_name_csv)
#     year_output.to_csv(output_file_path,compression=compression_opts,index=False)

#   output = pd.concat([output,year_output],ignore_index=True)

# output.convert_dtypes()

# game_data = pd.concat(list(year_game_data.values()))

# try:
#   game_data=game_data.drop(columns=[col for col in game_data.columns if 'Ferrell Center' in col])
#   output=output.drop(columns=[col for col in output.columns if 'Ferrell Center' in col])
# except:
#   pass

,Date,Team Rank,Opponent Rank,Opponent Name,Result,Possession Number,Location,Record,Conference,Postseason
0,Mon Nov 4,2,225,Maine,"W, 96-62",74,Home,1-0,,None
1,Fri Nov 8,4,312,Army,"W, 100-58",71,Home,2-0,,None
2,Tue Nov 12,3,16,Kentucky,"L, 77-72",75,Neutral,2-1,,None
3,Sat Nov 16,5,130,Wofford,"W, 86-35",64,Home,3-1,,None
4,Fri Nov 22,4,14,Arizona,"W, 69-55",68,Away,4-1,,None
5,Tue Nov 26,4,22,Kansas,"L, 75-72",68,Neutral,4-2,,None
6,Fri Nov 29,4,148,Seattle,"W, 70-48",68,Home,5-2,,None
7,Wed Dec 4,4,2,Auburn,"W, 84-78",63,Home,6-2,,None
8,Sun Dec 8,3,24,Louisville,"W, 76-65",67,Away,7-2,1-0,None
9,Tue Dec 10,3,250,Incarnate Word,"W, 72-46",62,Home,8-2,,None


#Data display

In [ ]:
#@title Previous games
YEAR = YEARS[0]
data = game_data[game_data.t1_name.str.contains(f'Auburn_2024')]
prev_games = data.iloc[-prev_games_to_avg:]
display(prev_games.head())

In [ ]:
#@title Previous Games Averages
output['date'] = pd.to_datetime(output['date'],utc=True,errors='raise')
output = output.drop(output[output['date']=='NaT'].index)
output = output.sort_values('date')
print(f"Most recent game: {datetime.fromisoformat(str(output['date'].iloc[-1])).ctime()}")
display(output[output.t1_name_left=='Auburn_2024'].tail())

In [ ]:
#@title Datasets
dataset = output.copy()
dataset = dataset.drop(['date','t1_name_left','t2_name_left','t2_name_left','t2_name_left','t1_name_right','t2_name_right','t2_name_right','t2_name_right'],axis=1).astype('float64')

train_dataset = dataset.sample(frac=0.8, random_state=0)
test_dataset = dataset.drop(train_dataset.index)
print('\n---Training dataset---')
display(train_dataset.head())
print('\n---Testing dataset---')
display(test_dataset.head())

In [ ]:
#@title Trainng features and labels
train_features = train_dataset.copy()
test_features = test_dataset.copy()

train_labels = pd.concat([train_features.pop(x) for x in ['pts_left', 'pts_right']], axis=1)
test_labels = pd.concat([test_features.pop(x) for x in ['pts_left', 'pts_right']], axis=1)

print('\n---Training features---')
display(train_features.head())
print('\n---Training labales---')
display(train_labels.head())

In [ ]:
#@title Dataset Information
dataset_info = train_dataset.describe().transpose()[['count','mean','std','min','max']]
setPandasDisplayLimit(None)
display(dataset_info)
setPandasDisplayLimit(10)

normalizer = tf.keras.layers.Normalization()
normalizer.adapt(np.array(train_features))

#Model Training

In [ ]:
#@title Model Parameters
model_save_name = "model" #@param {type: 'string'}
#@markdown Layer Architecture
num_layers_min =  3#@param {type: 'integer'}
num_layers_max =  5#@param {type: 'integer'}
num_layer_unit_min =  128#@param {type: 'integer'}
num_layer_unit_max = 1024 #@param {type: 'integer'}
num_layer_unit_step = 128 #@param {type: 'integer'}
num_layer_dropout = .25 #@param {type: 'number'}
num_layer_learning_rate_min = 1e-6 #@param {type: 'number'}
num_layer_learning_rate_max = 1e-1 #@param {type: 'number'}
num_layer_learning_rate_decay_rate = 1e-6 #@param {type: 'number'}
num_layer_learning_rate_decay_steps = 1000 #@param {type: 'integer'}

#@markdown Training Parameters
TRAIN_MODEL = False #@param {type: 'boolean'}
TUNE_NEW_MODEL = False #@param {type: 'boolean'}
training_epochs = 20000 #@param {type: 'integer'}
training_batch_size = 1024 #@param {type: 'integer'}
tuning_epochs = 200 #@param {type: 'integer'}
tuning_trials_max = 100 #@param {type: 'integer'}

test_results = {}

def build_model(hp):
  model = keras.Sequential()
  model.add(normalizer)
  for i in range(hp.Int("num_layers", num_layers_min, num_layers_max)):
        model.add(
            layers.Dense(
                # Tune number of units separately.
                units=hp.Int(f"units_{i}", min_value=num_layer_unit_min, max_value=num_layer_unit_max, step=num_layer_unit_step),
                activation=hp.Choice("activation", ["relu", "tanh",'sigmoid','selu']),
            )
        )
  if hp.Boolean("dropout"):
      model.add(layers.Dropout(rate=num_layer_dropout))
  model.add(layers.Dense(2))
  hp_learning_rate =  hp.Float("learning_rate", min_value=num_layer_learning_rate_min, max_value=num_layer_learning_rate_max, sampling="log")
  lr_schedule = keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=hp_learning_rate,
    decay_steps=num_layer_learning_rate_decay_steps,
    decay_rate=num_layer_learning_rate_decay_rate)
  loss_func = keras.losses.MeanAbsoluteError()
  model.compile(
    optimizer=tf.optimizers.Adam(learning_rate=lr_schedule),
    loss=loss_func)
  return model

def plot_loss(history):
  plt.plot(history.history['loss'], label='loss')
  plt.plot(history.history['val_loss'], label='val_loss')
  #plt.ylim([0, 20])
  plt.xlabel('Epoch')
  plt.ylabel('Error [pts]')
  plt.legend()
  plt.grid(True)

checkpoint_filepath = f'{folder_path}/{model_save_name}/checkpoint'
monitor_val = 'loss'
model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_filepath,
    monitor=monitor_val,
    mode='min',
    save_best_only=True,
    )
tqdm_callback = tfa.callbacks.TQDMProgressBar(
    show_epoch_progress=False,
    show_overall_progress=True,
    leave_epoch_progress=False,
    leave_overall_progress=True
)
stop_early = tf.keras.callbacks.EarlyStopping(
    monitor=monitor_val,
    mode='min',
    patience=500,
    restore_best_weights=True)

if TUNE_NEW_MODEL:
  tuner = kt.BayesianOptimization(build_model,
                     objective=kt.Objective(monitor_val, "min"),
                     max_trials=tuning_trials_max,
                     directory='my_dir',
                     project_name='sport_bot',
                     overwrite=True)
  tuner.search(train_features, train_labels, epochs=tuning_epochs, validation_split=0.2, callbacks=[stop_early])
  best_hps=tuner.get_best_hyperparameters(num_trials=1)[0]
  model = tuner.hypermodel.build(best_hps)

if TRAIN_MODEL:
  seed = np.random.RandomState()
  history = model.fit(
      train_features.sample(frac=1,random_state=seed),
      train_labels.sample(frac=1,random_state=seed),
      epochs=training_epochs,
      batch_size=training_batch_size,
      # Suppress logging.
      verbose=0,
      # Calculate validation results on 20% of the training data.
      validation_split = 0.2,
      callbacks=[model_checkpoint_callback,tqdm_callback])
  plot_loss(history)

#Model loading

In [ ]:
#@title Model Load Paramaters { run: "auto" }
model_load_name = "model_good_11"#@param {type: 'string'}
load_model_path = f"{folder_path}/{model_load_name}/checkpoint"
load = True #@param {type: 'boolean'}
if load:
  model = tf.keras.models.load_model(load_model_path)
  print(f'Loading model: {load_model_path}')
  print(model.summary())

In [ ]:
#@title Save model { run: "auto" }
new_model_save_name = "model_good_11"#@param {type: 'string'}
save = False #@param {type: 'boolean'}
if save:
  save_model_path = f"{folder_path}/{new_model_save_name}/checkpoint"
  model.save(save_model_path)

#Model Testing

In [ ]:
#@title Loss
avg_pts = pd.DataFrame(pd.DataFrame(np.full((test_labels.shape[0],test_labels.shape[1]),dataset_info['mean'][['pts_left','pts_right']]),index=test_labels.index))
loss_func = keras.losses.get(model.loss)
loss = loss_func(test_labels,avg_pts).numpy()
loss_df = pd.DataFrame([loss],columns=["Avg_Pts"])

test_results = model.evaluate(
    test_features, test_labels, verbose=0)
loss_df['Predicted_Pts'] = [test_results]
loss_df = loss_df.rename(index={0: loss_func.name})
display(loss_df)

In [ ]:
#@title Error Scatter plot
test_predictions = model.predict(test_features)

a = plt.axes(aspect='equal')
plt.scatter(test_labels, test_predictions)
plt.xlabel('True Values [pts]')
plt.ylabel('Predictions [pts]')
lims = [0, 150]
plt.xlim(lims)
plt.ylim(lims)
_ = plt.plot(lims, lims)

In [ ]:
#@title Error Histogram
error = np.mean(test_predictions - test_labels,axis=1)
plt.hist(error, bins=25)
plt.xlabel('Prediction Error [pts]')
_ = plt.ylabel('Count')

In [ ]:
#@title Prediction Winner Accuracy
import random
num_predict =  1000#@param {type: "integer"}
num_trials = 10 #@param {type: "integer"}

num_correct = 0

for i in tqdm(range(num_trials)):
  prediction_feature = test_features.sample(n=num_predict)
  actuals = test_labels.loc[prediction_feature.index]
  predictions = model.predict(x=prediction_feature)

  for j in range(predictions.shape[0]):
    prediction = predictions[j]
    actual = actuals.iloc[j].to_numpy()
    prediction_winner = "Team 1" if prediction[0] > prediction[1] else "Team 2"
    actual_winner = "Team 1" if actual[0] > actual[1] else "Team 2"
    if prediction_winner == actual_winner:
      num_correct+=1
    if num_trials-1 <= i and num_predict-1 <= j:
      print('---AVG Game Stats---')
      display(pd.DataFrame(prediction_feature.iloc[j]).T)
      print(f'\nPredicted pts: [{strformat.format(prediction[0])} - {strformat.format(prediction[1])}]')
      print(f"{prediction_winner} wins")
      print(f'\nActual pts: [{int(actual[0])} - {int(actual[1])}]')
      print(f"{actual_winner} wins")


print('\n---Model Winner Accuracy---')
print(f'\nNumber of Trials: {num_trials}')
print(f'Predictions: {num_predict}')

total_preditcions = (num_predict*num_trials)
print(f'Correct Predictions: {num_correct}')
print(f'Total Predictions: {total_preditcions}')
accuracy = (num_correct/total_preditcions) * 100
home_team_winner_accuracy = (game_data[(game_data.t1_pts > game_data.t2_pts) & (game_data['Home']==1)].shape[0] / game_data[(game_data['Home']==1)].shape[0]) * 100

print(f'\nHome Team Winner Accuracy: {strformat.format(home_team_winner_accuracy)}%')
print(f'Model Winner Accuracy: {strformat.format(accuracy)}%')


#Interactive Model

In [ ]:
#@title Game Parameters {display-mode: "form" }

#@markdown Team 1
team_1_name = 'Auburn' #@param {type:"string"}
team_1_year = 2019 #@param {type:"integer"}
team_1_rank =  12#@param {type:"integer"}
team_1_location = 'Neutral' #@param ["Home", "Away", "Neutral","Semi-Home","Semi-Away"]

#@markdown Team 2
team_2_name = 'Auburn' #@param {type:"string"}
team_2_year = 2024 #@param {type:"integer"}
team_2_rank =  4#@param {type:"integer"}
team_2_location = 'Neutral' #@param ["Home", "Away", "Neutral","Semi-Home","Semi-Away"]

def predictGame(teams):
  team_1_name = teams[0]['name']
  team_1_year = teams[0]['year']
  team_1_rank = teams[0]['rank']
  team_1_location = teams[0]['location']

  team_2_name = teams[1]['name']
  team_2_year = teams[1]['year']
  team_2_rank = teams[1]['rank']
  team_2_location = teams[1]['location']

  team_1_entry = f'{team_1_name}_{team_1_year}'
  team_2_entry = f'{team_2_name}_{team_2_year}'

  custom_output = game_data.copy()

  custom_output = custom_output.reindex(sorted(custom_output.columns), axis=1)

  team_1_features = custom_output[custom_output.t1_name == team_1_entry].iloc[-prev_games_to_avg:]
  team_1_feature_avg = pd.DataFrame(team_1_features.drop(['date','t1_name'],axis=1).agg('mean')).T
  team_1_feature_avg = team_1_feature_avg.rename(lambda x: x+'_avg', axis='columns')

  team_2_features = custom_output[custom_output.t1_name == team_2_entry].iloc[-prev_games_to_avg:]
  team_2_feature_avg = pd.DataFrame(team_2_features.drop(['date','t1_name'],axis=1).agg('mean')).T
  team_2_feature_avg = team_2_feature_avg.rename(lambda x: x+'_avg', axis='columns')


  team_1_feature_avg['t1_rank'] = team_1_rank
  team_1_feature_avg['t2_rank'] = team_2_rank
  team_2_feature_avg['t1_rank'] = team_2_rank
  team_2_feature_avg['t2_rank'] = team_1_rank


  team_1_feature_avg['Home'] = 1 if team_1_location == 'Home' else 0
  team_1_feature_avg['Away'] = 1 if team_1_location == 'Away' else 0
  team_1_feature_avg['Neutral'] = 1 if team_1_location == 'Neutral' else 0

  team_1_feature_avg['Semi-Home'] = 1 if team_1_location == 'Semi-Home' else 0
  team_1_feature_avg['Semi-Away'] = 1 if team_1_location == 'Semi-Away' else 0



  team_2_feature_avg['Home'] = 1 if team_2_location == 'Home' else 0
  team_2_feature_avg['Away'] = 1 if team_2_location == 'Away' else 0
  team_2_feature_avg['Neutral'] = 1 if team_2_location == 'Neutral' else 0


  team_2_feature_avg['Semi-Home'] = 1 if team_2_location == 'Semi-Away' else 0
  team_2_feature_avg['Semi-Away'] = 1 if team_2_location == 'Semi-Away' else 0

  features = team_1_feature_avg.merge(team_2_feature_avg, suffixes=('_left', '_right'),left_index=True,right_index=True)
  features = features.reindex(sorted(features.columns), axis=1)

  flipped_features = team_2_feature_avg.merge(team_1_feature_avg, suffixes=('_left', '_right'),left_index=True,right_index=True)
  flipped_features = flipped_features.reindex(sorted(flipped_features.columns), axis=1)
  pts = model.predict(features).flatten()
  flipped_pts = model.predict(flipped_features).flatten()

  avg_pts = [(pts[0] + flipped_pts[1])/2,(pts[1] + flipped_pts[0])/2]
  pts = avg_pts

  winner = team_1_name if pts[0] > pts[1] else team_2_name
  return  {'Team 1 Name': team_1_name,'Team 1 Year': team_1_year,'Team 1 Rank': team_1_rank,'Team 1 Location': team_1_location,'Team 1 Points': pts[0],'Team 2 Name': team_2_name,'Team 2 Year': team_2_year,'Team 2 Rank': team_2_rank,'Team 2 Location': team_2_location,'Team 2 Points': pts[1],'Winner': winner,'Spread': pts[0] - pts[1] if pts[1] > pts[0] else pts[1] - pts[0]}

def predictGames(games):
  predictions = []
  for game in games:
    teams = game['teams']

    team_1_name = teams[0]['name']
    team_1_year = teams[0]['year']
    team_1_rank = teams[0]['rank']
    team_1_location = teams[0]['location']

    team_2_name = teams[1]['name']
    team_2_year = teams[1]['year']
    team_2_rank = teams[1]['rank']
    team_2_location = teams[1]['location']
    try:
      predicted_data = predictGame(teams)
      if 'kp' in game and len(game['kp']["score"]) >= 2:
        prediction_kp = game['kp']
        predicted_data["Kenpom Winner"] = prediction_kp["winner"]
        predicted_data["Kenpom Team 1 Score"] = prediction_kp["score"][0]
        predicted_data["Kenpom Team 2 Score"] = prediction_kp["score"][1]
        predicted_data["Kenpom Spread"] = prediction_kp["spread"]
        predicted_data["Spread Diff"] = abs(predicted_data["Spread"] - prediction_kp["spread"])
        predicted_data["Spread Bet"] = "Over" if prediction_kp["spread"] > predicted_data["Spread"] else "Under"
      predictions.append(predicted_data)
    except Exception  as e:
      print(f"Cannot predict game for {team_1_name} vs {team_2_name} Error: ", e)
      continue
  return pd.DataFrame(predictions)


df = predictGames([{'teams':[{'name':team_1_name,'year':team_1_year,'rank':team_1_rank,'location':team_1_location},{'name':team_2_name,'year':team_2_year,'rank':team_2_rank,'location':team_2_location}]}])
display(df)


In [ ]:
#@title Predict Games on day
from datetime import date as dt
date = '2024-03-12' #@param {type: "string"}
skip = True #@param {type: "boolean"}

if not skip:
  games_url = f'{URL}/fanmatch.php?d={date}&s=Time'
  driver = init()
  login(driver)
  print(f'Retrieving games from: {games_url}')
  soup = goToURL(driver,games_url)

  games = []
  rows = [row for row in soup.find('tbody').find_all('tr') if len(row.contents) >=4]
  date_time = dt.fromisoformat(date)
  season_start = dt(date_time.year, 11, 1)
  season = date_time.year if date_time < season_start else date_time.year - 1
  print(season)

  for row in tqdm(rows):
    team_names = row.find_all(href=re.compile('team.php'))
    ranks = row.find_all('span',{'class' : ['seed','seed-gray']})
    ranks = row.find_all('span',{'class' : ['seed','seed-gray']})

    kp_predicted = row.find_all('td')[1].text
    ken_pom_prediction_winner =  re.search(r"\D+",kp_predicted)[0]
    ken_pom_prediction_score =  [int(score) for score in re.findall(r"\d+",kp_predicted)[0:2]]
    ken_pom_prediction_spread = 0
    if len(ken_pom_prediction_score) >= 2:
      ken_pom_prediction_spread = ken_pom_prediction_score[0] - ken_pom_prediction_score[1] if ken_pom_prediction_score[1] > ken_pom_prediction_score[0] else ken_pom_prediction_score[1] - ken_pom_prediction_score[0]
    location = row.find_all('td',style='text-align:left;')
    if not location:
      location = row.find_all('td',class_='bold-bottom-left')


    if len(team_names) >= 2:
      teams = [{},{}]
      teams[0]['name'] = team_names[0].text
      teams[1]['name'] = team_names[1].text

      teams[0]['rank'] = int(ranks[0].text)
      teams[1]['rank'] = int(ranks[1].text)

      teams[0]['year'] = season
      teams[1]['year'] = season

      teams[0]['location'] = 'Neutral'
      teams[1]['location'] = 'Neutral'

      location = location[-1].find(string=True,recursive=False).lower().strip()
      for i in range(len(team_names)):
        try:
          team_link = f"{URL}/{team_names[i]['href']}"
          team_page = goToURL(driver,team_link)
          arena_name = team_page.find('div',{'id':'title-container'}).find_all(string=True, recursive=False)[1].replace('\n','').split('·')[1].lower().strip()
          if arena_name == location:
            teams[i]['location'] = 'Home'
            teams[0 if i == 1 else 1]['location'] = 'Away'
            break
        except Exception  as e:
          print(f"Cannot determine location for {teams[0]['name']} vs {teams[1]['name']} ERROR: {str(e)}")
          pass

      games.append({'teams':teams,'kp': {'winner':ken_pom_prediction_winner,'score': ken_pom_prediction_score, "spread": ken_pom_prediction_spread}})
  setPandasDisplayLimit(None)
  display(predictGames(games))
  setPandasDisplayLimit(10)

In [ ]:
#@title Tournament Script
from dataclasses import dataclass
class Tournament:
    def __init__(self, rounds, calc_match_func):
        self.round = 0
        self.done = False
        self.calc_match_func = calc_match_func
        self.tournament_winner = None
        self.rounds = []
        for round in rounds:
            matches = []
            for match in round:
                matches.append(Match(match))
            self.rounds.append(matches)

    def add_round(self, matches, round=-1):
        if round == -1:
            round = self.round + 1
        self.rounds.insert(round, matches)
    def calculate_round(self):
        next_round = []
        if self.round+1 < len(self.rounds):
            next_round.extend(self.rounds[self.round+1])
        for index in range(len(self.rounds[self.round])):
            match = self.rounds[self.round][index]
            match.winner = self.calc_match_winner(match, self.calc_match_func)
            winner = match.teams[match.winner]
            found = False
            for next_round_match in next_round:
                if len(next_round_match.teams) < 2:
                    next_round_match.teams.append(winner)
                    found = True
                    break
            if not found:
                next_round.append(Match([match.teams[match.winner]], 0))
        self.round = self.round + 1
        if self.round < len(self.rounds):
            self.rounds[self.round] = next_round
        else:
            self.rounds.append(next_round)
        return next_round

    def calc_match_winner(self,match, func):
        return func(match.teams)

    def sim_tournament(self):
        while not self.done:
            print(f'\n\n===========================Round {self.round}===========================')
            self.calculate_round()
            if len(self.rounds[self.round]) <= 1 and len(self.rounds[self.round][0].teams)<= 1:
                self.done = True
        self.tournament_winner = self.rounds[self.round][0].teams[self.rounds[self.round][0].winner]
@dataclass
class Match:
    teams: list[str]
    winner: int = 0

def get_team_data(team_names_list):
  print(f'Retrieving Team Data')
  team_names_list_f = []
  for round in team_names_list:
    team_names_list_f.extend(np.hstack(round))
  teams_data = {}
  for team_name in team_names_list_f:
    team_url = f'https://kenpom.com/team.php?team={team_name.replace(" ","+").replace("&","%26").strip()}&y={str(year)}'
    soup = goToURL(driver,team_url)
    title = soup.find('div',id='title-container')
    try:
      header = title.h5
    except Exception as e:
      print(f"Cannot find header for {team_name} ERROR: {str(e)}")
      return None
    record_info = header.find_all('span',class_='rank')
    rank = record_info[0].text
    name = team_name.strip()
    teams_data[name] = {'rank':mk_int(rank),'year':year}
  return teams_data

def calc_winner(names_list):
  teams = []
  games = []
  for team_name in names_list:
    team = {}
    team['name'] = team_name.strip()
    team['rank'] = team_data[team['name']]['rank']
    team['year'] = team_data[team['name']]['year']
    team['location'] = 'Neutral'
    teams.append(team)
  games.append({'teams':teams})
  pred = predictGames(games)
  setPandasDisplayLimit(None)
  display(pred)
  setPandasDisplayLimit(10)
  return names_list.index(pred['Winner'][0])

In [ ]:
#@title Predict Tournament
team_names_list = [     [         ["Connecticut","Stetson"],         ["Florida Atlantic","Northwestern"],         ["San Diego St.","UAB"],         ["Auburn","Yale"],         ["BYU","Duquesne"],         ["Illinois","Morehead St."],         ["Washington St.","Drake"],         ["Iowa St.","South Dakota St."],          ["North Carolina","Wagner"],         ["Mississippi St.","Michigan St."],         ["Saint Mary's","Grand Canyon"],         ["Alabama","Charleston"],         ["Clemson","New Mexico"],         ["Baylor","Colgate"],         ["Dayton","Nevada"],         ["Arizona","Long Beach St."],          ["Houston","Longwood"],         ["Nebraska","Texas A&M"],         ["Wisconsin","James Madison"],         ["Duke","Vermont"],         ["Texas Tech","N.C. State"],         ["Kentucky","Oakland"],         ["Florida","Colorado"],         ["Marquette","Western Kentucky"],          ["Purdue","Grambling St."],         ["Utah St.","TCU"],         ["Gonzaga","McNeese St."],         ["Kansas","Samford"],         ["South Carolina","Oregon"],         ["Creighton","Akron"],         ["Texas","Colorado St."],         ["Tennessee","Saint Peter's"]     ] ] #@param {type: "raw"}
year = 2025 #@param {type: "number"}
print(f'Siming tourny games')

driver = init()
login(driver)
team_data = get_team_data(team_names_list)
if team_data is not None:
  tb = Tournament(team_names_list, calc_winner)
  tb.sim_tournament()
  print(f'Tournament Winner: {tb.tournament_winner}')




In [ ]:
#@title Predict Games
team_names_list = [["Auburn" ,"Alabama"]] #@param {type: "raw"}

print(f'Siming {len(team_names_list)} games')

driver = init()
login(driver)
year = 2024
games = []
for names_list in team_names_list:
  teams = []
  for team_name in names_list:
    team_url = f'https://kenpom.com/team.php?team={team_name.replace(" ","+").replace("&","%26").strip()}&y={str(year)}'
    print(f'Retrieving Team Data from: {team_url}')

    soup = goToURL(driver,team_url)
    title = soup.find('div',id='title-container')
    header = title.h5
    record_info = header.find_all('span',class_='rank')
    rank = record_info[0].text

    team = {}
    team['name'] = team_name.strip()
    team['rank'] = mk_int(rank)
    team['year'] = year
    team['location'] = 'Neutral'
    teams.append(team)
  games.append({'teams':teams})
setPandasDisplayLimit(None)
display(predictGames(games))
setPandasDisplayLimit(10)